# Paper Extraction
Extract and only get a specific range of date and id paper

In [ ]:
import json
import os
import pandas as pd

INPUT_PATH = "/kaggle/input/datasets/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json"

START_DATE, START_ID = "2408", 16045
END_DATE, END_ID = "2409", 3593

OUTPUT_JSONL = f"/kaggle/working/arxiv_{START_DATE}_{START_ID}_to_{END_DATE}_{END_ID}.jsonl"
OUTPUT_PARQUET = f"/kaggle/working/arxiv_{START_DATE}_{START_ID}_to_{END_DATE}_{END_ID}.parquet"

# Sanity check on the input path
if not os.path.exists(INPUT_PATH):
    found = [os.path.join(r, f) for r, _, fs in os.walk("/kaggle/input") for f in fs]
    raise FileNotFoundError(f"{INPUT_PATH} not found. Files available:\n" + "\n".join(found[:20]))


def month_range(start_ym: str, end_ym: str) -> list[str]:
    """All YYMM strings from start to end inclusive."""
    y, m = divmod(int(start_ym), 100)
    ey, em = divmod(int(end_ym), 100)
    out = []
    
    while (y, m) <= (ey, em):
        out.append(f"{y:02d}{m:02d}")
        m += 1
        if m > 12:
            y, m = y + 1, 1
    return out


# Fast pre-filter prefixes, e.g. '"2408.' and '"2409.' (no closing quote!)
PREFILTER = tuple(f'"{ym}.' for ym in month_range(START_DATE, END_DATE))


def is_in_target_range(arxiv_id: str) -> bool:
    ym, sep, num_str = arxiv_id.partition(".")
    # New-style IDs only: YYMM.NNNNN (skips old IDs like "hep-th/9901001")
    if not sep or len(ym) != 4 or not ym.isdigit() or not num_str.isdigit():
        return False
    return (int(START_DATE), START_ID) <= (int(ym), int(num_str)) <= (int(END_DATE), END_ID)


matched_records = []

with open(INPUT_PATH, "r", encoding="utf-8") as infile, \
     open(OUTPUT_JSONL, "w", encoding="utf-8") as outfile:

    for line in infile:
        if not any(p in line for p in PREFILTER):
            continue

        record = json.loads(line)
        paper_id = str(record.get("id", ""))

        if is_in_target_range(paper_id):
            record["title"] = " ".join(str(record.get("title", "")).split())
            record["abstract"] = " ".join(str(record.get("abstract", "")).split())

            matched_records.append(record)
            outfile.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Total papers found in range [{START_DATE}.{START_ID:05d} -> {END_DATE}.{END_ID:05d}]: {len(matched_records)}")

if matched_records:
    df_slice = pd.DataFrame(matched_records)
    df_slice.to_parquet(OUTPUT_PARQUET, index=False)
    print(f"Saved {OUTPUT_JSONL} and {OUTPUT_PARQUET} successfully!")
else:
    print("No records matched; skipping parquet. Check INPUT_PATH and the ID range.")